In [2]:
from pathlib import Path
import pickle
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from config import RESULTS_DIR

DATA_DIR = Path(RESULTS_DIR) / "random_trajectories"
CHECKPOINT_DIR = Path(RESULTS_DIR) / "transformer"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "hexapod_transformer_fitness.pt"

SEED = 42
BATCH_SIZE = 32
EPOCHS = 80
LEARNING_RATE = 1e-4
TRAIN_SPLIT = 0.8
MAX_SEQ_LEN = 2000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")
print(f"Loading trajectories from: {DATA_DIR}")


Using device: cuda
Loading trajectories from: /home/joze/Mestrado/Dissertacao/Pybullet/results/random_trajectories


In [3]:
class RandomTrajectoryFitnessDataset(Dataset):
    """Lazy dataset that keeps only file paths in memory and loads sequences on demand.

    Assumes one trajectory per file (dict or list-with-one-entry). This avoids loading
    tens of thousands of pickles into RAM at once which caused the kernel OOM/crash.
    """

    def __init__(self, file_paths):
        self.file_paths = list(file_paths)
        if not self.file_paths:
            raise RuntimeError('No trajectory files provided')

        # Determine length and sample a single file to infer state_dim
        self._infer_state_dim()

    def _load_entry_from_file(self, path):
        try:
            with open(path, "rb") as f:
                loaded = pickle.load(f)
        except (EOFError, pickle.UnpicklingError):
            print(f"Skipping corrupted file: {path}")
            return None

        entries = loaded if isinstance(loaded, list) else [loaded]

        for entry in entries:

            # New nested format
            if "result" in entry:
                entry = entry["result"]

            if (
                "detailed_log" in entry
                and entry.get("fitness") is not None
            ):
                return entry

        return None

    def _infer_state_dim(self):
        for p in self.file_paths:
            entry = self._load_entry_from_file(p)
            if entry is None:
                continue
            detailed_log = entry.get('detailed_log', [])
            if len(detailed_log) == 0:
                continue
            step = detailed_log[0]
            position = np.asarray(step.get('position', []), dtype=np.float32)
            orientation = np.asarray(step.get('orientation', []), dtype=np.float32)
            contacts = np.asarray(step.get('contacts', []), dtype=np.float32)
            torques = np.asarray(step.get('torques', []), dtype=np.float32)
            self.state_dim = int(position.size + orientation.size + contacts.size + torques.size)
            return
        # Fallback
        self.state_dim = 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        entry = self._load_entry_from_file(path)
        if entry is None:
            # Return an empty sequence with zero fitness (shouldn't happen for valid dataset)
            return np.zeros((1, self.state_dim), dtype=np.float32), np.float32(0.0)

        detailed_log = entry.get('detailed_log', [])
        fitness = float(entry.get('fitness', 0.0))

        seq = []
        for step in detailed_log:
            position = np.asarray(step.get('position', []), dtype=np.float32)
            orientation = np.asarray(step.get('orientation', []), dtype=np.float32)
            contacts = np.asarray(step.get('contacts', []), dtype=np.float32)
            torques = np.asarray(step.get('torques', []), dtype=np.float32)
            state = np.concatenate([position, orientation, contacts, torques]).astype(np.float32)
            seq.append(state)

        if not seq:
            return np.zeros((1, self.state_dim), dtype=np.float32), np.float32(fitness)

        seq = np.stack(seq)
        return seq, np.float32(fitness)


def load_random_trajectory_files(data_dir: Path):
    return sorted(data_dir.glob('traj_*.pkl'))


def compute_feature_stats_sampled(file_paths, sample_count=500):
    """Compute mean/std by randomly sampling up to `sample_count` files and accumulating state statistics.

    This avoids loading the entire corpus into memory.
    """
    file_paths = list(file_paths)
    n_files = len(file_paths)
    if n_files == 0:
        raise RuntimeError('No files to compute stats from')

    sample_count = min(sample_count, n_files)
    sampled = random.sample(file_paths, sample_count)

    sum_ = None
    sumsq = None
    total = 0

    for p in sampled:
        with open(p, 'rb') as f:
            loaded = pickle.load(f)
        entries = loaded if isinstance(loaded, list) else [loaded]
        entry = None
        entry = None

        for e in entries:

            # New nested format
            if "result" in e:
                e = e["result"]

            if "detailed_log" in e and e.get("fitness") is not None:
                entry = e
                break

        if entry is None:
            continue
        seq = []
        for step in entry['detailed_log']:
            position = np.asarray(step.get('position', []), dtype=np.float64)
            orientation = np.asarray(step.get('orientation', []), dtype=np.float64)
            contacts = np.asarray(step.get('contacts', []), dtype=np.float64)
            torques = np.asarray(step.get('torques', []), dtype=np.float64)
            state = np.concatenate([position, orientation, contacts, torques]).astype(np.float64)
            seq.append(state)
        if not seq:
            continue
        seq = np.stack(seq, axis=0)  # (T, D)
        if sum_ is None:
            D = seq.shape[1]
            sum_ = np.zeros(D, dtype=np.float64)
            sumsq = np.zeros(D, dtype=np.float64)
        sum_ += seq.sum(axis=0)
        sumsq += (seq ** 2).sum(axis=0)
        total += seq.shape[0]

    if total == 0:
        raise RuntimeError('No timesteps found while sampling files for stats')

    mean = sum_ / total
    var = (sumsq / total) - (mean ** 2)
    std = np.sqrt(np.maximum(var, 1e-12))
    std[std < 1e-6] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)


# collate_fn factory remains the same and will be used below


In [4]:
class HexapodTransformer(nn.Module):
    def __init__(self, state_dim, d_model=64, nhead=4, num_layers=4, dim_feedforward=256, dropout=0.1, max_seq_len=2000):
        super().__init__()
        self.input_proj = nn.Linear(state_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len + 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fitness_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1),
        )

    def forward(self, states, padding_mask=None):
        x = self.input_proj(states)
        batch_size, seq_len, _ = x.shape

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        if seq_len + 1 > self.pos_embedding.shape[1]:
            raise ValueError(
                f'Sequence length {seq_len} exceeds max_seq_len {self.pos_embedding.shape[1] - 1}'
            )

        x = x + self.pos_embedding[:, :seq_len + 1]

        if padding_mask is not None:
            cls_mask = torch.zeros((padding_mask.shape[0], 1), dtype=torch.bool, device=padding_mask.device)
            padding_mask = torch.cat([cls_mask, padding_mask], dim=1)

        x = self.transformer(x, src_key_padding_mask=padding_mask)
        cls_output = x[:, 0]
        fitness_pred = self.fitness_head(cls_output).squeeze(-1)
        return fitness_pred, cls_output


In [5]:
trajectory_files = load_random_trajectory_files(DATA_DIR)
if not trajectory_files:
    raise FileNotFoundError(f'No pickle files found in {DATA_DIR}')

print(f'Found {len(trajectory_files)} trajectory files')

# lazy dataset keeps only file paths
dataset = RandomTrajectoryFitnessDataset(trajectory_files)
print(f'Loaded {len(dataset)} trajectories')
print(f'State dimension: {dataset.state_dim}')

train_size = max(1, int(len(dataset) * TRAIN_SPLIT))
val_size = len(dataset) - train_size
if val_size == 0:
    val_size = 1
    train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

# Compute feature stats by sampling files from the training subset (dataset is lazy)
train_indices = getattr(train_dataset, 'indices', None)
if train_indices is None:
    # Fallback: use first `train_size` files
    train_file_paths = trajectory_files[:train_size]
else:
    train_file_paths = [dataset.file_paths[i] for i in train_indices]

feature_mean, feature_std = compute_feature_stats_sampled(train_file_paths, sample_count=500)

# Inline collate_fn using sampled mean/std (avoid dependency on previous cell state)
feature_mean_t = torch.tensor(feature_mean, dtype=torch.float32)
feature_std_t = torch.tensor(feature_std, dtype=torch.float32)

def collate_fn(batch):
    sequences, fitness = zip(*batch)
    lengths = torch.tensor([seq.shape[0] for seq in sequences], dtype=torch.long)
    max_len = int(lengths.max().item())
    feature_dim = sequences[0].shape[-1]

    states = torch.zeros((len(sequences), max_len, feature_dim), dtype=torch.float32)
    padding_mask = torch.ones((len(sequences), max_len), dtype=torch.bool)

    for i, seq in enumerate(sequences):
        seq_tensor = torch.tensor(seq, dtype=torch.float32)
        seq_tensor = (seq_tensor - feature_mean_t) / feature_std_t
        seq_len = seq_tensor.shape[0]
        states[i, :seq_len] = seq_tensor
        padding_mask[i, :seq_len] = False

    fitness = torch.tensor(fitness, dtype=torch.float32)
    return states, padding_mask, lengths, fitness

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Checkpoint path: {CHECKPOINT_PATH}')


Found 11239 trajectory files
Loaded 11239 trajectories
State dimension: 31
Train samples: 8991
Validation samples: 2248
Checkpoint path: /home/joze/Mestrado/Dissertacao/Pybullet/results/transformer/hexapod_transformer_fitness.pt


In [7]:
model = HexapodTransformer(
    state_dim=dataset.state_dim,
    d_model=64,
    nhead=4,
    num_layers=4,
    dim_feedforward=256,
    dropout=0.1,
    max_seq_len=MAX_SEQ_LEN,
).to(DEVICE)

In [ ]:


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

best_val_loss = float('inf')
best_epoch = -1
epochs_without_improvement = 0
patience = 5
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for states, padding_mask, lengths, fitness_target in train_loader:
        states = states.to(DEVICE)
        padding_mask = padding_mask.to(DEVICE)
        fitness_target = fitness_target.to(DEVICE)

        pred, _ = model(states, padding_mask=padding_mask)
        loss = loss_fn(pred, fitness_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * states.size(0)

    # --------------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for states, padding_mask, lengths, fitness_target in val_loader:
            states = states.to(DEVICE)
            padding_mask = padding_mask.to(DEVICE)
            fitness_target = fitness_target.to(DEVICE)

            pred, _ = model(states, padding_mask=padding_mask)
            loss = loss_fn(pred, fitness_target)

            val_loss += loss.item() * states.size(0)

    train_loss /= len(train_dataset)
    val_loss /= len(val_dataset)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss
    })

    print(
        f'Epoch {epoch:03d} | '
        f'train_loss={train_loss:.6f} | '
        f'val_loss={val_loss:.6f}'
    )

    # --------------------------------------------------------------
    # CHECK FOR IMPROVEMENT
    # --------------------------------------------------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'state_dim': dataset.state_dim,
                'feature_mean': feature_mean,
                'feature_std': feature_std,
                'best_val_loss': best_val_loss,
                'best_epoch': best_epoch,
                'history': history,
                'config': {
                    'd_model': 64,
                    'nhead': 4,
                    'num_layers': 4,
                    'dim_feedforward': 256,
                    'dropout': 0.1,
                    'max_seq_len': MAX_SEQ_LEN,
                },
            },
            CHECKPOINT_PATH,
        )

        print('  -> New best model saved.')

    else:
        epochs_without_improvement += 1

        print(
            f'  -> No improvement '
            f'({epochs_without_improvement}/{patience})'
        )

    # --------------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------------
    if epochs_without_improvement >= patience:
        print(
            f'\nEarly stopping at epoch {epoch}. '
            f'Validation loss did not improve for {patience} '
            f'consecutive epochs.'
        )
        break


print(
    f'Best validation loss: {best_val_loss:.6f} '
    f'at epoch {best_epoch}'
)

/home/joze/Mestrado/Dissertacao/Pybullet/bulletenv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 001 | train_loss=0.026653 | val_loss=0.011317
  -> New best model saved.
Epoch 002 | train_loss=0.011610 | val_loss=0.004978
  -> New best model saved.
Epoch 003 | train_loss=0.007386 | val_loss=0.003670
  -> New best model saved.
Epoch 004 | train_loss=0.005914 | val_loss=0.003743
  -> No improvement (1/5)
Epoch 005 | train_loss=0.005055 | val_loss=0.003284
  -> New best model saved.
Epoch 006 | train_loss=0.004751 | val_loss=0.002996
  -> New best model saved.
Epoch 007 | train_loss=0.004059 | val_loss=0.002995
  -> New best model saved.
Epoch 008 | train_loss=0.003610 | val_loss=0.002766
  -> New best model saved.
Epoch 009 | train_loss=0.003295 | val_loss=0.002203
  -> New best model saved.
Epoch 010 | train_loss=0.003037 | val_loss=0.002166
  -> New best model saved.
Epoch 011 | train_loss=0.002659 | val_loss=0.001590
  -> New best model saved.
Epoch 012 | train_loss=0.002330 | val_loss=0.001472
  -> New best model saved.
Epoch 013 | train_loss=0.002096 | val_loss=0.001335
 

In [9]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
print(f"Saved transformer checkpoint to: {CHECKPOINT_PATH}")
print(f"Checkpoint keys: {sorted(checkpoint.keys())}")

sample_states, sample_mask, sample_lengths, sample_target = next(iter(val_loader))
model.eval()
with torch.no_grad():
    sample_pred, sample_latent = model(sample_states.to(DEVICE), padding_mask=sample_mask.to(DEVICE))

print(f"Example predictions: {sample_pred[:5].detach().cpu().numpy()}")
print(f"Example targets: {sample_target[:5].numpy()}")
print(f"Latent shape: {tuple(sample_latent.shape)}")


Saved transformer checkpoint to: /home/joze/Mestrado/Dissertacao/Pybullet/results/transformer/hexapod_transformer_fitness.pt
Checkpoint keys: ['best_epoch', 'best_val_loss', 'config', 'feature_mean', 'feature_std', 'history', 'model_state_dict', 'state_dim']
Example predictions: [1.1074136  0.5333878  0.8544712  0.6536995  0.39786965]
Example targets: [0.1666886  0.22750638 0.03149763 0.02024144 0.39462778]
Latent shape: (32, 64)


In [10]:
from sklearn.decomposition import PCA

model.eval()

latent_vectors = []

with torch.no_grad():
    for states, padding_mask, lengths, fitness in train_loader:

        states = states.to(DEVICE)
        padding_mask = padding_mask.to(DEVICE)

        _, cls_output = model(states, padding_mask)

        latent_vectors.append(
            cls_output.cpu().numpy()
        )

latent_vectors = np.concatenate(latent_vectors, axis=0)

print(latent_vectors.shape)

(8991, 64)


In [11]:
pca = PCA(n_components=2)

pca_latents = pca.fit_transform(latent_vectors)

print(pca.explained_variance_ratio_)

with open("transformer_pca.pkl", "wb") as f:
    pickle.dump(pca, f)

[0.24229658 0.1908183 ]


In [12]:
# Load PCA and transformer checkpoint, build model, and expose helper to compute 2D BD
from pathlib import Path
import pickle

# Locate PCA file (try common locations)
possible_pca_paths = [
    Path(CHECKPOINT_DIR) / 'transformer_pca.pkl',
    Path(RESULTS_DIR) / 'transformer' / 'transformer_pca.pkl',
    Path('transformer_pca.pkl'),
]

pca_path = None
for p in possible_pca_paths:
    if p is not None and p.exists():
        pca_path = p
        break

if pca_path is None:
    raise FileNotFoundError('transformer_pca.pkl not found in expected locations: ' + ','.join(str(p) for p in possible_pca_paths))

with open(pca_path, 'rb') as f:
    pca = pickle.load(f)

# Load checkpoint (weights_only=False to allow saved numpy metadata)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

model = HexapodTransformer(
    state_dim=checkpoint['state_dim'],
    **checkpoint['config']
).to(DEVICE)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Normalization stats saved in checkpoint
feature_mean = np.asarray(checkpoint.get('feature_mean'))
feature_std = np.asarray(checkpoint.get('feature_std'))


def compute_transformer_bd(trajectory_states):
    """Compute 2D behavior descriptor from a raw trajectory states array.

    Args:
        trajectory_states: numpy array of shape (T, state_dim) matching the training features
    Returns:
        bd_2d: numpy array shape (2,) projected by PCA
    """
    # Normalize using recorded training stats
    states_norm = (trajectory_states.astype(np.float32) - feature_mean.astype(np.float32)) / feature_std.astype(np.float32)

    states_tensor = torch.tensor(states_norm, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        # model returns (fitness_pred, cls_output)
        _, cls_output = model(states_tensor)

    latent = cls_output.cpu().numpy()[0]

    bd_2d = pca.transform(latent.reshape(1, -1))[0]

    return bd_2d

print(f'Loaded PCA from: {pca_path} and transformer checkpoint from: {CHECKPOINT_PATH}')


Loaded PCA from: transformer_pca.pkl and transformer checkpoint from: /home/joze/Mestrado/Dissertacao/Pybullet/results/transformer/hexapod_transformer_fitness.pt


In [13]:
print(pca_latents.min(axis=0))
print(pca_latents.max(axis=0))


[-4.0567575 -4.101915 ]
[4.779473 3.915781]


Fine Tuning

In [1]:
model = HexapodTransformer(
    state_dim=dataset.state_dim,
    d_model=64,
    nhead=4,
    num_layers=4,
    dim_feedforward=256,
    dropout=0.1,
    max_seq_len=MAX_SEQ_LEN,
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])

loss_fn = nn.MSELoss()

print(f"Loaded checkpoint from epoch {checkpoint['best_epoch']}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.6f}")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,      # 10x smaller
    weight_decay=1e-2,
)

best_val_loss = float("inf")
best_epoch = 0
history = checkpoint["history"]
epochs_without_improvement = 0
patience = 5

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for states, padding_mask, lengths, fitness_target in train_loader:
        states = states.to(DEVICE)
        padding_mask = padding_mask.to(DEVICE)
        fitness_target = fitness_target.to(DEVICE)

        pred, _ = model(states, padding_mask=padding_mask)
        loss = loss_fn(pred, fitness_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * states.size(0)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for states, padding_mask, lengths, fitness_target in val_loader:
            states = states.to(DEVICE)
            padding_mask = padding_mask.to(DEVICE)
            fitness_target = fitness_target.to(DEVICE)

            pred, _ = model(states, padding_mask=padding_mask)
            loss = loss_fn(pred, fitness_target)
            val_loss += loss.item() * states.size(0)

    train_loss /= len(train_dataset)
    val_loss /= len(val_dataset)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

    print(f'Epoch {epoch:03d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "state_dim": dataset.state_dim,
                "feature_mean": feature_mean,
                "feature_std": feature_std,
                "best_val_loss": best_val_loss,
                "best_epoch": epoch,
                "history": history,
                "config": checkpoint["config"],
            },
            CHECKPOINT_PATH,
        )

    else:
        epochs_without_improvement += 1

        print(
            f'  -> No improvement '
            f'({epochs_without_improvement}/{patience})'
        )

    # --------------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------------
    if epochs_without_improvement >= patience:
        print(
            f'\nEarly stopping at epoch {epoch}. '
            f'Validation loss did not improve for {patience} '
            f'consecutive epochs.'
        )
        break

print(f'Best validation loss: {best_val_loss:.6f} at epoch {best_epoch}')

NameError: name 'HexapodTransformer' is not defined

In [24]:
from pathlib import Path
import pickle

bad_files = []

for p in Path(DATA_DIR).glob("traj_*.pkl"):
    try:
        with open(p, "rb") as f:
            pickle.load(f)
    except Exception:
        bad_files.append(p)

print(f"{len(bad_files)} corrupted files found.")

for p in bad_files[:10]:
    print(p)

5 corrupted files found.
/home/joze/Mestrado/Dissertacao/Pybullet/results/random_trajectories/traj_5c082d28f0604480943364020b5c21d5.pkl
/home/joze/Mestrado/Dissertacao/Pybullet/results/random_trajectories/traj_28c8069bea9749e6af8e1add70168849.pkl
/home/joze/Mestrado/Dissertacao/Pybullet/results/random_trajectories/traj_8f53c21576d342e48b45b0f43bd4bdf7.pkl
/home/joze/Mestrado/Dissertacao/Pybullet/results/random_trajectories/traj_cca444663ad8420d8363a340427e57a6.pkl
/home/joze/Mestrado/Dissertacao/Pybullet/results/random_trajectories/traj_b9fdd6ebd28e43fc9fc02db33642a7fe.pkl


In [36]:
import pickle
from pathlib import Path
import numpy as np

# Change this to one of your trajectory files
traj_file = Path(DATA_DIR) / "traj_fffdc7e18adf49899f83759c5cdfca45.pkl"

with open(traj_file, "rb") as f:
    data = pickle.load(f)

print("=" * 80)
print("Type:", type(data))

# Some files may contain a list with one entry
if isinstance(data, list):
    print(f"Number of entries: {len(data)}")
    entry = data[0]
else:
    entry = data

print("\nTop-level keys:")
print(entry.keys())

print("\nFitness:", entry.get("fitness"))

detailed_log = entry.get("detailed_log", [])
print("Trajectory length:", len(detailed_log))

print(data.keys())

print("\nresult:")
print(type(data["result"]))
print(data["result"].keys())

print("\nbehavior_descriptors:")
print(type(data["behavior_descriptors"]))
print(data["behavior_descriptors"])


print(data["result"]["fitness"])
print(len(data["result"]["detailed_log"]))
print(data["result"]["detailed_log"][0].keys())


if len(detailed_log) > 0:
    print("\nKeys in first timestep:")
    print(detailed_log[0].keys())

    first = detailed_log[0]

    for key, value in first.items():
        arr = np.asarray(value)
        print(f"{key:15s} shape={arr.shape} dtype={arr.dtype}")
    

print("=" * 80)

Type: <class 'dict'>

Top-level keys:
dict_keys(['result', 'behavior_descriptors'])

Fitness: None
Trajectory length: 0
dict_keys(['result', 'behavior_descriptors'])

result:
<class 'dict'>
dict_keys(['fitness', 'behavior_descriptors', 'final_position', 'distance', 'energy', 'detailed_log'])

behavior_descriptors:
<class 'numpy.ndarray'>
[-0.04873061 -0.2388703 ]
0.24756264894192115
60
dict_keys(['time', 'position', 'orientation', 'contacts', 'torques'])
